In [1]:
%pip install pandas 

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install datetime

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd 
from datetime import datetime 

#data upload 

file=r"D:\project-police_stop_report\traffic_stops.csv"
df=pd.read_csv(file)


C:\Users\Iniya R\AppData\Local\Temp\ipykernel_42276\1487806060.py:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(file)


In [6]:
df

,stop_date,stop_time,country_name,driver_gender,driver_age_raw,driver_age,driver_race,violation_raw,violation,search_conducted,search_type,stop_outcome,is_arrested,stop_duration,drugs_related_stop,vehicle_number
0,2020-01-01,0:00:00,Canada,M,59,19,Asian,Drunk Driving,Speeding,True,Vehicle Search,Ticket,True,16-30 Min,True,UP76DY3473
1,2020-01-01,0:01:00,India,M,35,58,Other,Other,Other,False,Vehicle Search,Arrest,True,16-30 Min,True,RJ83PZ4441
2,2020-01-01,0:02:00,USA,M,26,76,Black,Signal Violation,Speeding,False,Frisk,Ticket,True,16-30 Min,True,RJ32OM7264
3,2020-01-01,0:03:00,Canada,M,26,76,Black,Speeding,DUI,True,Frisk,Warning,False,0-15 Min,True,RJ76TI3807
4,2020-01-01,0:04:00,Canada,M,62,75,Other,Speeding,Other,False,Vehicle Search,Arrest,True,16-30 Min,False,WB63BB8305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65533,2020-02-15,12:13:00,India,F,54,48,Black,Other,Other,False,Vehicle Search,Arrest,True,16-30 Min,False,DL56GW6568
65534,2020-02-15,12:14:00,Canada,F,18,35,Hispanic,Seatbelt,Other,True,Vehicle Search,Ticket,False,16-30 Min,True,TN73EO7098
65535,2020-02-15,12:15:00,USA,M,27,41,Asian,Seatbelt,DUI,True,Frisk,Ticket,True,30+ Min,True,GJ33MX8328
65536,2020-02-15,12:16:00,Canada,F,49,63,Black,Seatbelt,Other,False,NaN,Warning,True,0-15 Min,True,KA24UZ8488


In [9]:
import pandas as pd
import mysql.connector
from datetime import datetime


# ✅ Clean data
df.dropna(axis=1, how='all', inplace=True)
df.fillna({
    'driver_age': df['driver_age'].median(),
    'search_type': 'None',
    'stop_duration': 'Unknown',
    'violation': 'Unknown',
    'stop_outcome': 'Unknown',
    'driver_gender': 'Unknown',
    'driver_race': 'Unknown',
    'country_name': 'Unknown'
}, inplace=True)

# ✅ Create timestamp
df['timestamp'] = pd.to_datetime(df['stop_date'] + ' ' + df['stop_time'], errors='coerce')
df['timestamp'].fillna(datetime.now(), inplace=True)

C:\Users\Iniya R\AppData\Local\Temp\ipykernel_42276\289377154.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['timestamp'].fillna(datetime.now(), inplace=True)


In [11]:
import mysql.connector

connection = mysql.connector.connect(
    host="gateway01.ap-southeast-1.prod.aws.tidbcloud.com",
    port=4000,
    user="3k51nZsQpw5fDsi.root",
    password="wPp6beRvkSzwGtZQ",
  
)
mycursor = connection.cursor(buffered=True)

In [13]:
mycursor.execute("create database project")

In [14]:
mycursor.execute("USE project ")

In [15]:
# Drop tables if they exist
for table in ['vehicles', 'violation', 'check_post_logs']:
    mycursor.execute(f"DROP TABLE IF EXISTS {table}")

# Create vehicles table
mycursor.execute("""
CREATE TABLE vehicles (
    vehicle_number VARCHAR(20) PRIMARY KEY,
    driver_gender VARCHAR(10),
    driver_age INT,
    driver_race VARCHAR(20),
    timestamp TIMESTAMP,
    status VARCHAR(50)
)
""")

# Create violations table
mycursor.execute("""
CREATE TABLE violation (
    vehicle_number VARCHAR(20),
    violation_type VARCHAR(100),
    drugs_related_stop BOOLEAN
)
""")

# Create check_post_logs table
mycursor.execute("""
CREATE TABLE check_post_logs (
    vehicle_number VARCHAR(20),
    stop_outcome VARCHAR(50),
    stop_duration VARCHAR(50)
)
""")

connection.commit()
print("✅ Tables created successfully!")

✅ Tables created successfully!


In [16]:
# Prepare batch insert for vehicles
vehicles_data = [
    (
        row['vehicle_number'],
        row['driver_gender'],
        int(row['driver_age']),
        row['driver_race'],
        row['timestamp'],
        'Flagged' if row.get('drugs_related_stop', False) else 'Clear'
    )
    for _, row in df.iterrows()
]

mycursor.executemany("""
    INSERT IGNORE INTO vehicles (vehicle_number, driver_gender, driver_age, driver_race, timestamp, status)
    VALUES (%s, %s, %s, %s, %s, %s)
""", vehicles_data)
print(f"✅ Inserted {len(vehicles_data)} vehicles")

# -------------------------
# Prepare batch insert for violations
violations_data = [
    (
        row['vehicle_number'],
        row['violation'],
        bool(row['drugs_related_stop'])
    )
    for _, row in df.iterrows()
]

mycursor.executemany("""
    INSERT INTO violation (vehicle_number, violation_type, drugs_related_stop)
    VALUES (%s, %s, %s)
""", violations_data)
print(f"✅ Inserted {len(violations_data)} violations")

# -------------------------
# Prepare batch insert for check_post_logs
logs_data = [
    (
        row['vehicle_number'],
        row['stop_outcome'],
        row['stop_duration']
    )
    for _, row in df.iterrows()
]

mycursor.executemany("""
    INSERT INTO check_post_logs (vehicle_number, stop_outcome, stop_duration)
    VALUES (%s, %s, %s)
""", logs_data)
print(f"✅ Inserted {len(logs_data)} logs")

# Commit once for all inserts
connection.commit()
mycursor.close()
connection.close()
print("🎯 All CSV data inserted successfully!")

✅ Inserted 65538 vehicles
✅ Inserted 65538 violations
✅ Inserted 65538 logs
🎯 All CSV data inserted successfully!
